In [ ]:
!pip install ultralytics opencv-python supervision

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# ============================================================
# VEHICLE COUNTING USING YOLO11 + BYTETRACK
# ============================================================
#
# FINAL COUNTING LOGIC
#
# PINK -> YELLOW
#     = TOWARDS CAMERA / DOWNWARDS
#
# YELLOW -> PINK
#     = AWAY FROM CAMERA / UPWARDS
#
# IMPORTANT:
# - Starts processing from FRAME 1
# - Processes EVERY frame
# - Uses vehicle CENTER point internally
# - Vehicle must cross BOTH lines
# - Count occurs when SECOND line is crossed
# - Motorcycle is counted exactly like other vehicles
# - Lines moved UP to capture more vehicles
# - Separate counter for every vehicle type
# - BOUNDING BOXES / CENTER POINTS / ID LABELS HIDDEN
# ============================================================


# ============================================================
# 2. IMPORTS
# ============================================================

import cv2
import os

from ultralytics import YOLO

from collections import defaultdict

from IPython.display import (
    display,
    Video,
    clear_output
)


# ============================================================
# 3. FILE PATHS
# ============================================================

INPUT_VIDEO = "/content/video.mp4"

OUTPUT_VIDEO = "/content/vehicle_counting_output.mp4"

MODEL_PATH = "yolo11l.pt"

TRACKER_CONFIG = "/content/bytetrack_custom.yaml"


# ============================================================
# 4. CUSTOM BYTETRACK CONFIGURATION
# ============================================================

tracker_yaml = """
tracker_type: bytetrack
track_high_thresh: 0.15
track_low_thresh: 0.05
new_track_thresh: 0.15
track_buffer: 60
match_thresh: 0.80
fuse_score: True
"""


with open(
    TRACKER_CONFIG,
    "w"
) as f:

    f.write(
        tracker_yaml
    )


print(
    "ByteTrack configuration created."
)


# ============================================================
# 5. LOAD YOLO MODEL
# ============================================================

model = YOLO(
    MODEL_PATH
)


# ============================================================
# 6. VEHICLE CLASSES
# ============================================================

VEHICLE_CLASSES = {

    1: "Bicycle",

    2: "Car",

    3: "Motorcycle",

    5: "Bus",

    7: "Truck"
}


VEHICLE_CLASS_IDS = list(
    VEHICLE_CLASSES.keys()
)


# ============================================================
# 7. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(
    INPUT_VIDEO
)


if not cap.isOpened():

    raise Exception(
        "Could not open input video."
    )


# ============================================================
# 8. VIDEO INFORMATION
# ============================================================

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)


height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)


fps = cap.get(
    cv2.CAP_PROP_FPS
)


if fps <= 0:

    fps = 30


total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


print("\n========================================")
print("VIDEO INFORMATION")
print("========================================")

print(
    "Width       :",
    width
)

print(
    "Height      :",
    height
)

print(
    "FPS         :",
    fps
)

print(
    "Total frames:",
    total_frames
)


# ============================================================
# 9. COUNTING LINE POSITIONS
# ============================================================

PINK_LINE_Y = int(height * 0.50)

YELLOW_LINE_Y = int(height * 0.65)


print("\n========================================")
print("UPDATED COUNTING LINES")
print("========================================")

print(
    "Pink line   :",
    PINK_LINE_Y
)

print(
    "Yellow line :",
    YELLOW_LINE_Y
)


# ============================================================
# 10. CREATE OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)


out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (
        width,
        height
    )
)


if not out.isOpened():

    raise Exception(
        "Could not create output video."
    )


# ============================================================
# 11. VEHICLE STATE
# ============================================================

vehicle_state = {}


# ============================================================
# 12. PREVIOUS CENTER POSITION
# ============================================================

previous_y = {}


# ============================================================
# 13. COUNTED VEHICLES
# ============================================================

counted_down_ids = set()

counted_up_ids = set()


# ============================================================
# 14. COUNTERS
# ============================================================

towards_count = defaultdict(
    int
)

away_count = defaultdict(
    int
)


# ============================================================
# 15. CLASS VOTING
# ============================================================

class_scores = defaultdict(
    lambda: defaultdict(float)
)


def update_class_score(
    track_id,
    vehicle_name,
    confidence
):

    class_scores[
        track_id
    ][
        vehicle_name
    ] += confidence


def get_best_class(
    track_id
):

    if track_id not in class_scores:

        return "Unknown"


    if not class_scores[
        track_id
    ]:

        return "Unknown"


    return max(
        class_scores[
            track_id
        ],
        key=class_scores[
            track_id
        ].get
    )


# ============================================================
# 16. COUNTER PANEL
# ============================================================

def draw_counter_panel(
    frame
):

    panel_x = 15
    panel_y = 15

    panel_width = 315
    panel_height = 220


    # --------------------------------------------------------
    # TRANSPARENT BACKGROUND
    # --------------------------------------------------------

    overlay = frame.copy()


    cv2.rectangle(

        overlay,

        (
            panel_x,
            panel_y
        ),

        (
            panel_x + panel_width,
            panel_y + panel_height
        ),

        (20, 20, 20),

        -1

    )


    frame = cv2.addWeighted(

        overlay,

        0.72,

        frame,

        0.28,

        0

    )


    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    cv2.putText(

        frame,

        "VEHICLE COUNT",

        (
            panel_x + 10,
            panel_y + 23
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    # --------------------------------------------------------
    # HEADERS
    # --------------------------------------------------------

    cv2.putText(

        frame,

        "VEHICLE",

        (
            panel_x + 10,
            panel_y + 46
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.37,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        "TOWARDS",

        (
            panel_x + 125,
            panel_y + 46
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.36,

        (0, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        "AWAY",

        (
            panel_x + 240,
            panel_y + 46
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.36,

        (255, 105, 180),

        1,

        cv2.LINE_AA

    )


    # --------------------------------------------------------
    # VEHICLE TYPES
    # --------------------------------------------------------

    vehicles = [

        "Car",

        "Motorcycle",

        "Bus",

        "Truck",

        "Bicycle"

    ]


    y = panel_y + 70


    for vehicle in vehicles:

        # Vehicle name
        cv2.putText(

            frame,

            vehicle,

            (
                panel_x + 10,
                y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.37,

            (255, 255, 255),

            1,

            cv2.LINE_AA

        )


        # Towards
        cv2.putText(

            frame,

            str(
                towards_count[
                    vehicle
                ]
            ),

            (
                panel_x + 145,
                y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.42,

            (0, 255, 255),

            1,

            cv2.LINE_AA

        )


        # Away
        cv2.putText(

            frame,

            str(
                away_count[
                    vehicle
                ]
            ),

            (
                panel_x + 255,
                y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.42,

            (255, 105, 180),

            1,

            cv2.LINE_AA

        )


        y += 25


    # --------------------------------------------------------
    # TOTAL
    # --------------------------------------------------------

    total_towards = sum(
        towards_count.values()
    )


    total_away = sum(
        away_count.values()
    )


    cv2.putText(

        frame,

        "TOTAL",

        (
            panel_x + 10,
            y + 2
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.40,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        str(
            total_towards
        ),

        (
            panel_x + 145,
            y + 2
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.45,

        (0, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        str(
            total_away
        ),

        (
            panel_x + 255,
            y + 2
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.45,

        (255, 105, 180),

        1,

        cv2.LINE_AA

    )


    return frame


# ============================================================
# 17. INITIALIZE VEHICLE
# ============================================================

def initialize_vehicle(
    track_id,
    cy
):

    if cy < PINK_LINE_Y:

        vehicle_state[
            track_id
        ] = "POTENTIAL_DOWN"


    elif cy > YELLOW_LINE_Y:

        vehicle_state[
            track_id
        ] = "POTENTIAL_UP"


    else:

        vehicle_state[
            track_id
        ] = "BETWEEN"


# ============================================================
# 18. PROCESS VIDEO FROM FRAME 1
# ============================================================

frame_number = 0


while True:

    # ========================================================
    # READ FRAME
    # ========================================================

    ret, frame = cap.read()


    if not ret:

        break


    frame_number += 1


    # ========================================================
    # YOLO + BYTETRACK
    # ========================================================

    results = model.track(

        frame,

        persist=True,

        tracker=TRACKER_CONFIG,

        classes=VEHICLE_CLASS_IDS,

        conf=0.35,

        iou=0.50,

        imgsz=1280,

        verbose=False

    )


    # ========================================================
    # DRAW PINK LINE
    # ========================================================

    cv2.line(

        frame,

        (
            0,
            PINK_LINE_Y
        ),

        (
            width,
            PINK_LINE_Y
        ),

        (255, 105, 180),

        3

    )


    cv2.putText(

        frame,

        "PINK",

        (
            10,
            PINK_LINE_Y - 10
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255, 105, 180),

        1,

        cv2.LINE_AA

    )


    # ========================================================
    # DRAW YELLOW LINE
    # ========================================================

    cv2.line(

        frame,

        (
            0,
            YELLOW_LINE_Y
        ),

        (
            width,
            YELLOW_LINE_Y
        ),

        (0, 255, 255),

        3

    )


    cv2.putText(

        frame,

        "YELLOW",

        (
            10,
            YELLOW_LINE_Y - 10
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (0, 255, 255),

        1,

        cv2.LINE_AA

    )


    # ========================================================
    # CHECK TRACKING RESULTS
    # ========================================================

    if (

        results

        and

        results[0].boxes is not None

        and

        results[0].boxes.id is not None

    ):

        boxes = results[0].boxes


        track_ids = (

            boxes.id
            .int()
            .cpu()
            .tolist()

        )


        class_ids = (

            boxes.cls
            .int()
            .cpu()
            .tolist()

        )


        confidences = (

            boxes.conf
            .cpu()
            .tolist()

        )


        coordinates = (

            boxes.xyxy
            .cpu()
            .tolist()

        )


        # ====================================================
        # PROCESS EACH VEHICLE
        # ====================================================

        for (

            track_id,

            class_id,

            confidence,

            box

        ) in zip(

            track_ids,

            class_ids,

            confidences,

            coordinates

        ):


            # ------------------------------------------------
            # VEHICLE FILTER
            # ------------------------------------------------

            if class_id not in VEHICLE_CLASSES:

                continue


            vehicle_name = VEHICLE_CLASSES[
                class_id
            ]


            # ------------------------------------------------
            # BOUNDING BOX
            # ------------------------------------------------

            x1, y1, x2, y2 = map(
                int,
                box
            )


            # ------------------------------------------------
            # CENTER
            # ------------------------------------------------
            # Center is STILL calculated internally.
            # It is NOT drawn on the video.
            # ------------------------------------------------

            cx = (
                x1 + x2
            ) // 2


            cy = (
                y1 + y2
            ) // 2


            # ------------------------------------------------
            # CLASS SCORE
            # ------------------------------------------------

            update_class_score(

                track_id,

                vehicle_name,

                confidence

            )


            best_class = get_best_class(
                track_id
            )


            # =================================================
            # NEW TRACK
            # =================================================

            if track_id not in previous_y:

                previous_y[
                    track_id
                ] = cy


                initialize_vehicle(

                    track_id,

                    cy

                )


                # =================================================
                # BOUNDING BOX / CENTER / LABEL
                # ARE INTENTIONALLY NOT DRAWN
                # =================================================

                continue


            # =================================================
            # PREVIOUS CENTER
            # =================================================

            old_y = previous_y[
                track_id
            ]


            # =================================================
            # MOVEMENT
            # =================================================

            movement = (
                cy - old_y
            )


            # =================================================
            # ACTUAL CROSSING
            # =================================================

            pink_down = (

                old_y
                <
                PINK_LINE_Y

                and

                cy
                >=
                PINK_LINE_Y

            )


            pink_up = (

                old_y
                >
                PINK_LINE_Y

                and

                cy
                <=
                PINK_LINE_Y

            )


            yellow_down = (

                old_y
                <
                YELLOW_LINE_Y

                and

                cy
                >=
                YELLOW_LINE_Y

            )


            yellow_up = (

                old_y
                >
                YELLOW_LINE_Y

                and

                cy
                <=
                YELLOW_LINE_Y

            )


            # =================================================
            # INITIAL STATE
            # =================================================

            current_state = vehicle_state[
                track_id
            ]


            # =================================================
            # VEHICLE STARTED ABOVE PINK
            # =================================================

            if current_state == "POTENTIAL_DOWN":

                if pink_down:

                    vehicle_state[
                        track_id
                    ] = "PINK_FIRST"


            # =================================================
            # VEHICLE STARTED BELOW YELLOW
            # =================================================

            elif current_state == "POTENTIAL_UP":

                if yellow_up:

                    vehicle_state[
                        track_id
                    ] = "YELLOW_FIRST"


            # =================================================
            # VEHICLE STARTED BETWEEN LINES
            # =================================================

            elif current_state == "BETWEEN":

                if movement > 0:

                    vehicle_state[
                        track_id
                    ] = "PINK_FIRST"


                elif movement < 0:

                    vehicle_state[
                        track_id
                    ] = "YELLOW_FIRST"


            # =================================================
            # PINK FIRST
            # WAIT FOR YELLOW
            # =================================================

            elif current_state == "PINK_FIRST":

                if yellow_down:

                    # ----------------------------------------
                    # SECOND LINE CROSSED
                    #
                    # PINK -> YELLOW
                    #
                    # COUNT TOWARDS
                    # ----------------------------------------

                    if track_id not in counted_down_ids:

                        counted_down_ids.add(
                            track_id
                        )


                        vehicle_state[
                            track_id
                        ] = "COUNTED_DOWN"


                        final_class = get_best_class(
                            track_id
                        )


                        if final_class in VEHICLE_CLASSES.values():

                            towards_count[
                                final_class
                            ] += 1


                            print(
                                f"[TOWARDS/DOWN] "
                                f"ID={track_id} "
                                f"{final_class} "
                                f"PINK->YELLOW"
                            )


            # =================================================
            # YELLOW FIRST
            # WAIT FOR PINK
            # =================================================

            elif current_state == "YELLOW_FIRST":

                if pink_up:

                    # ----------------------------------------
                    # SECOND LINE CROSSED
                    #
                    # YELLOW -> PINK
                    #
                    # COUNT AWAY
                    # ----------------------------------------

                    if track_id not in counted_up_ids:

                        counted_up_ids.add(
                            track_id
                        )


                        vehicle_state[
                            track_id
                        ] = "COUNTED_UP"


                        final_class = get_best_class(
                            track_id
                        )


                        if final_class in VEHICLE_CLASSES.values():

                            away_count[
                                final_class
                            ] += 1


                            print(
                                f"[AWAY/UP] "
                                f"ID={track_id} "
                                f"{final_class} "
                                f"YELLOW->PINK"
                            )


            # =================================================
            # UPDATE PREVIOUS Y
            # =================================================

            previous_y[
                track_id
            ] = cy


            # =================================================
            # NO BOUNDING BOX / CENTER / LABEL DRAWING
            # =================================================
            #
            # Tracking and counting continue internally.
            #
            # The following visual elements are intentionally
            # hidden:
            #
            # cv2.rectangle()
            # cv2.circle()
            # cv2.putText() vehicle label
            #
            # =================================================


    # ========================================================
    # DRAW COUNTER
    # ========================================================

    frame = draw_counter_panel(
        frame
    )


    # ========================================================
    # SAVE FRAME
    # ========================================================

    out.write(
        frame
    )


    # ========================================================
    # SMALL LIVE PREVIEW
    # ========================================================

    if frame_number % 30 == 0:

        preview_width = 800


        scale = (
            preview_width
            /
            width
        )


        preview_height = int(
            height * scale
        )


        small_frame = cv2.resize(

            frame,

            (
                preview_width,
                preview_height
            )

        )


        small_frame_rgb = cv2.cvtColor(

            small_frame,

            cv2.COLOR_BGR2RGB

        )


        clear_output(
            wait=True
        )


        display(
            small_frame_rgb
        )


        total_towards = sum(
            towards_count.values()
        )


        total_away = sum(
            away_count.values()
        )


        print(
            f"Processing frame "
            f"{frame_number}/{total_frames}"
        )


        print(
            f"Progress: "
            f"{frame_number / total_frames * 100:.1f}%"
        )


        print(
            f"Towards / Down: "
            f"{total_towards}"
        )


        print(
            f"Away / Up: "
            f"{total_away}"
        )


# ============================================================
# 19. RELEASE
# ============================================================

cap.release()

out.release()


# ============================================================
# 20. FINAL COUNTS
# ============================================================

total_towards = sum(
    towards_count.values()
)


total_away = sum(
    away_count.values()
)


print("\n")
print("========================================")
print("PROCESSING COMPLETE")
print("========================================")


# ============================================================
# TOWARDS
# ============================================================

print("\n")
print("TOWARDS CAMERA / DOWNWARDS")
print("PINK -> YELLOW")
print("----------------------------------------")


for vehicle in [

    "Car",

    "Motorcycle",

    "Bus",

    "Truck",

    "Bicycle"

]:

    print(

        f"{vehicle:<12}: "
        f"{towards_count[vehicle]}"

    )


print(
    f"{'TOTAL':<12}: "
    f"{total_towards}"
)


# ============================================================
# AWAY
# ============================================================

print("\n")
print("AWAY FROM CAMERA / UPWARDS")
print("YELLOW -> PINK")
print("----------------------------------------")


for vehicle in [

    "Car",

    "Motorcycle",

    "Bus",

    "Truck",

    "Bicycle"

]:

    print(

        f"{vehicle:<12}: "
        f"{away_count[vehicle]}"

    )


print(
    f"{'TOTAL':<12}: "
    f"{total_away}"
)


# ============================================================
# 21. OUTPUT VIDEO
# ============================================================

print("\n")
print("========================================")
print("OUTPUT VIDEO")
print("========================================")


print(
    OUTPUT_VIDEO
)


print(
    "File exists:",
    os.path.exists(
        OUTPUT_VIDEO
    )
)


# ============================================================
# 22. DISPLAY COMPLETE VIDEO
# ============================================================

print(
    "\nDisplaying complete processed video..."
)


display(

    Video(

        OUTPUT_VIDEO,

        embed=True,

        width=800

    )

)


# ============================================================
# 23. DOWNLOAD
# ============================================================

from google.colab import files


files.download(
    OUTPUT_VIDEO
)